# 🖥️ Computer Vision Lab 8 — Convolutional Neural Networks on MNIST

**BITS F459 — Computer Vision**

Runnable companion to [`labsheet8.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet8.md). The sheet carries the theory;
this notebook carries the code, in order.

Sections 1–3 use `%%writefile` to create `img_dataset.py`, `cnn_model.py` and
`img_train.py` as **real files on disk**. **Lab 9 reuses two of them unchanged**, so keep
this folder.

> 📌 This lab answers the question Lab 6-2 ended on: the flat MLP's ceiling on MNIST is
> "the limitation of treating an image as an unstructured vector of numbers". Convolution
> removes that limitation.

> ⚠️ **Check your kernel** — this needs **Python (cv-dl)**. If cell 1 raises
> `ModuleNotFoundError`, you are on the wrong kernel (Lab 6-1 Section 0).

---
## 0️⃣ Setup

In [ ]:
import sys, torch, torchvision
print("python     :", sys.executable)        # must contain 'cv-dl'
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA       :", torch.cuda.is_available())   # False is fine

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

---
## 1️⃣ Convolution Arithmetic

Output size for one spatial dimension:

```
    out = floor( (n + 2p − d·(k − 1) − 1) / s ) + 1
```

Two cases worth memorising: `k=3, p=1, s=1` preserves size; `k=3, p=1, s=2` halves it.

In [ ]:
import torch
from torch import nn
for k, p, s in [(3, 1, 1), (3, 0, 1), (3, 1, 2), (5, 2, 1), (5, 0, 1), (7, 3, 1)]:
    out = nn.Conv2d(1, 1, k, stride=s, padding=p)(torch.zeros(1, 1, 28, 28))
    print(k, p, s, "->", tuple(out.shape[-2:]))

### The parameter-sharing win

A conv layer's parameter count is `out_ch × (in_ch × k × k) + out_ch` — and it **does not
depend on the image size at all**. A dense layer's does.

In [ ]:
from torch import nn

conv = nn.Conv2d(1, 32, 3, padding=1)
dense = nn.Linear(28 * 28, 32)
pc = sum(p.numel() for p in conv.parameters())
pd = sum(p.numel() for p in dense.parameters())
print(f"Conv2d(1,32,3) : {pc:7d}")
print(f"Linear(784,32) : {pd:7d}   ({pd / pc:.1f}x more)")

# Now imagine a 224x224 input. Which number changes?
print(f"Linear(224*224,32): {224 * 224 * 32 + 32:7d}   Conv2d is unchanged at {pc}")

### Write `img_dataset.py`

Chronology matters here as it did in Lab 7: this file keeps the `(1, 28, 28)` spatial
layout that convolution needs, and provides both **augmentation** (fresh random shift each
epoch) and a **fixed shifted test set** for the honest robustness measurement in 5.4.

> ⚠️ **Fill shifted images with the normalized black value, not literal zero.** After
> normalisation a black pixel is `(0 − 0.1307)/0.3081 ≈ −0.424`. Padding with `0` paints a
> grey frame the network can detect, and it will learn "grey border ⇒ shifted" instead of
> becoming robust.

In [ ]:
%%writefile img_dataset.py
"""img_dataset.py - MNIST as images, with augmentation and a shifted test set.

Lab 6 flattened MNIST to 784-vectors. Convolutions need the spatial layout,
so everything here keeps the (1, 28, 28) shape. Lab 9 reuses this file.
"""

import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision.datasets import MNIST

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081     # the standard MNIST statistics


def load_mnist_arrays(root="./data", train=True):
    """Returns (images uint8 (N, 28, 28), targets int64).

    torchvision exposes .data and .targets directly, so we skip PIL entirely.
    Falls back to OpenML if the mirrors are down - the same guard as Lab 6-2,
    because yann.lecun.com now 404s permanently.
    """
    try:
        base = MNIST(root=root, train=train, download=True)
        return base.data.numpy(), base.targets.clone()
    except Exception as exc:
        print(f"[mnist] torchvision download failed ({type(exc).__name__}); "
              f"falling back to OpenML...")
        from sklearn.datasets import fetch_openml
        X, y = fetch_openml("mnist_784", version=1, return_X_y=True,
                            as_frame=False, parser="auto")
        X = X.astype(np.uint8).reshape(-1, 28, 28)
        y = y.astype(np.int64)
        span = slice(0, 60000) if train else slice(60000, 70000)
        return X[span], torch.tensor(y[span])


class MNISTImageDataset(Dataset):
    """MNIST as normalized (1, 28, 28) float tensors.

    shift_max : if > 0, every image is randomly translated by up to this many
                pixels when it is fetched. Used BOTH as training augmentation
                and (with a fixed seed) to build a shifted TEST set, which is
                how Section 5.4 measures translation robustness honestly.
    """

    def __init__(self, root="./data", train=True, limit=None, shift_max=0,
                 fixed_shift=False, random_state=42):
        images, targets = load_mnist_arrays(root=root, train=train)
        x = torch.tensor(images, dtype=torch.float32) / 255.0
        x = (x - MNIST_MEAN) / MNIST_STD
        self.images = x[:, None]                     # (N, 1, 28, 28)
        self.targets = targets
        self.shift_max = shift_max
        self.fixed_shift = fixed_shift

        if limit is not None and limit < len(self.targets):
            g = torch.Generator().manual_seed(random_state)
            keep = torch.randperm(len(self.targets), generator=g)[:limit]
            self.images, self.targets = self.images[keep], self.targets[keep]

        if shift_max > 0 and fixed_shift:
            # Apply ONCE, deterministically: a fixed, reproducible test set.
            g = torch.Generator().manual_seed(random_state)
            shifts = torch.randint(-shift_max, shift_max + 1,
                                   (len(self.targets), 2), generator=g)
            self.images = torch.stack([
                _translate(img, int(dx), int(dy))
                for img, (dx, dy) in zip(self.images, shifts)
            ])

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.shift_max > 0 and not self.fixed_shift:
            # Fresh random shift each epoch -> this is AUGMENTATION.
            dx, dy = torch.randint(-self.shift_max, self.shift_max + 1, (2,))
            img = _translate(img, int(dx), int(dy))
        return img, self.targets[idx]


def _translate(img, dx, dy):
    """Shift a (1, H, W) tensor by (dx, dy) pixels, filling with the
    normalized value of a black pixel (NOT zero - see the warning below)."""
    pad = (0 - MNIST_MEAN) / MNIST_STD
    out = torch.full_like(img, pad)
    h, w = img.shape[-2:]
    xs_src = slice(max(0, -dx), min(w, w - dx))
    xs_dst = slice(max(0, dx), min(w, w + dx))
    ys_src = slice(max(0, -dy), min(h, h - dy))
    ys_dst = slice(max(0, dy), min(h, h + dy))
    out[:, ys_dst, xs_dst] = img[:, ys_src, xs_src]
    return out

---
## 2️⃣ `cnn_model.py`

`receptive_field` implements `RF = 1 + Σ (k_i − 1) · Π_{j<i} s_j`. `ManualConv2d` is the
from-scratch version, in the tradition of Lab 7's `ManualLSTM`.

In [ ]:
%%writefile cnn_model.py
"""cnn_model.py - a configurable CNN, a from-scratch conv, and RF arithmetic."""

import torch
import torch.nn.functional as F
from torch import nn


def receptive_field(layers):
    """Receptive field of one output unit, for a stack of (kernel, stride) pairs.

    RF = 1 + sum_i (k_i - 1) * prod_{j<i} s_j
    Pooling counts as a layer too: maxpool 2x2 stride 2 is (2, 2).
    """
    rf, jump = 1, 1
    for k, s in layers:
        rf += (k - 1) * jump
        jump *= s
    return rf


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class SimpleCNN(nn.Module):
    """(B, 1, 28, 28) -> (B, num_classes).

    channels   : one conv block per entry, e.g. (32, 64)
    norm       : "none" | "batch"
    downsample : "max" | "avg" | "stride" | "none"
    head       : "flatten" (classic) | "gap" (global average pool)
    Returns raw logits. Softmax at display time only - as in Labs 6 and 7.
    """

    def __init__(self, in_channels=1, num_classes=10, channels=(32, 64),
                 kernel_size=3, norm="none", downsample="max", dropout=0.0,
                 head="flatten", head_hidden=128, input_size=28):
        super().__init__()
        self.layer_spec = []                       # for receptive_field()
        blocks, prev = [], in_channels
        for ch in channels:
            stride = 2 if downsample == "stride" else 1
            pad = kernel_size // 2                 # "same" padding for odd k
            blocks.append(nn.Conv2d(prev, ch, kernel_size, stride=stride, padding=pad))
            self.layer_spec.append((kernel_size, stride))
            if norm == "batch":
                blocks.append(nn.BatchNorm2d(ch))
            blocks.append(nn.ReLU())
            if downsample == "max":
                blocks.append(nn.MaxPool2d(2)); self.layer_spec.append((2, 2))
            elif downsample == "avg":
                blocks.append(nn.AvgPool2d(2)); self.layer_spec.append((2, 2))
            prev = ch
        self.features = nn.Sequential(*blocks)

        # Work out the flattened width by running one dummy tensor through,
        # rather than deriving it by hand and getting it wrong after an edit.
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, input_size, input_size)
            feat = self.features(dummy)
        self.feature_shape = tuple(feat.shape[1:])

        if head == "gap":
            # Global average pool: one number per channel, so the head size is
            # independent of the input resolution.
            self.pool = nn.AdaptiveAvgPool2d(1)
            n_flat = prev
        elif head == "flatten":
            self.pool = nn.Identity()
            n_flat = int(torch.tensor(self.feature_shape).prod())
        else:
            raise ValueError(f"unknown head {head!r}")

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(n_flat, head_hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

    def feature_maps(self, x):
        """Activations after every block - for the visualisation in 5.5."""
        maps, h = [], x
        for layer in self.features:
            h = layer(h)
            if isinstance(layer, nn.ReLU):
                maps.append(h.detach())
        return maps


class FlatMLP(nn.Module):
    """Lab 6's MLP, taking the SAME (B, 1, 28, 28) input so the comparison
    in Section 4 is controlled: identical data, identical training loop."""

    def __init__(self, num_classes=10, hidden=(128,), dropout=0.0, input_size=28):
        super().__init__()
        layers, prev = [nn.Flatten()], input_size * input_size
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ---------------------------------------------------------------- from scratch
class ManualConv2d(nn.Module):
    """A 2-D convolution written out with unfold, to show what Conv2d does.

    Parameter layout mirrors nn.Conv2d, so you can copy weights across and
    check the outputs agree - the same exercise as Lab 7's ManualLSTM.

    The trick: im2col. F.unfold rips out every k*k patch and lays it in a
    column, turning convolution into one big matrix multiply.
    """

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.k, self.stride, self.padding = kernel_size, stride, padding
        self.out_channels = out_channels
        scale = 1.0 / (in_channels * kernel_size * kernel_size) ** 0.5
        self.weight = nn.Parameter(
            torch.empty(out_channels, in_channels, kernel_size, kernel_size).uniform_(-scale, scale))
        self.bias = nn.Parameter(torch.empty(out_channels).uniform_(-scale, scale))

    def forward(self, x):
        B, _, H, W = x.shape
        out_h = (H + 2 * self.padding - self.k) // self.stride + 1
        out_w = (W + 2 * self.padding - self.k) // self.stride + 1
        cols = F.unfold(x, self.k, padding=self.padding, stride=self.stride)
        out = self.weight.view(self.out_channels, -1) @ cols      # (B, out_ch, L)
        out = out + self.bias[None, :, None]
        return out.view(B, self.out_channels, out_h, out_w)

### Verify `ManualConv2d` against `nn.Conv2d`

If this passes, your im2col convolution is exactly PyTorch's.

In [ ]:
import torch
from torch import nn
from cnn_model import ManualConv2d, SimpleCNN, receptive_field, count_params

torch.manual_seed(0)
ref = nn.Conv2d(3, 8, 3, stride=2, padding=1)
man = ManualConv2d(3, 8, 3, stride=2, padding=1)
man.weight.data.copy_(ref.weight.data)
man.bias.data.copy_(ref.bias.data)

x = torch.randn(4, 3, 28, 28)
a, b = ref(x), man(x)
print("shapes:", tuple(a.shape), tuple(b.shape))
print("allclose:", torch.allclose(a, b, atol=1e-5))

### Receptive fields and feature shapes

In [ ]:
print("receptive fields:")
print("  three 3x3, stride 1 :", receptive_field([(3, 1)] * 3))
print("  conv3 -> pool2 -> conv3:", receptive_field([(3, 1), (2, 2), (3, 1)]))

print()
for ds in ("max", "avg", "stride", "none"):
    m = SimpleCNN(channels=(32, 64), downsample=ds)
    print(f"  downsample={ds:7s} feature_shape={m.feature_shape} "
          f"params={count_params(m):8d} RF={receptive_field(m.layer_spec)}")

print()
for head in ("flatten", "gap"):
    m = SimpleCNN(channels=(32, 64), head=head)
    conv = sum(p.numel() for p in m.features.parameters())
    clf = sum(p.numel() for p in m.classifier.parameters())
    print(f"  head={head:8s} conv={conv:7d} head={clf:7d} total={count_params(m):8d}")

> 🔎 **Look at that last block before you believe any "CNNs are small" claim.** With a
> `flatten` head the classifier holds the overwhelming majority of the parameters; the
> convolution itself is almost free. `gap` inverts that — at a cost measured in Section 4.

---
## 3️⃣ `img_train.py`

Same shape as Lab 7's `seq_train.py`. Two additions: an optional cosine LR schedule, and
`add_images` logging so you can *see* what the model is being fed — augmentation bugs are
far easier to spot in a picture than in a number.

> 💡 **`model=` is the extension point.** `build_model` knows only `"cnn"` and `"mlp"`, but
> `train()` accepts a ready-made module — which is exactly how **Lab 9 reuses this file**
> for a Vision Transformer.

In [ ]:
%%writefile img_train.py
"""img_train.py - one parameterized training run for image models."""

import os
import time

import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from cnn_model import SimpleCNN, FlatMLP, count_params


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_model(config):
    """Everything in Labs 8 and 9 goes through here, so a run is fully
    described by its config dict - which is what makes the experiment
    tables in Section 5 honest."""
    if config["model"] == "mlp":
        return FlatMLP(num_classes=config.get("num_classes", 10),
                       hidden=config.get("hidden", (128,)),
                       dropout=config.get("dropout", 0.0))
    if config["model"] == "cnn":
        return SimpleCNN(
            num_classes=config.get("num_classes", 10),
            channels=tuple(config.get("channels", (32, 64))),
            kernel_size=config.get("kernel_size", 3),
            norm=config.get("norm", "none"),
            downsample=config.get("downsample", "max"),
            dropout=config.get("dropout", 0.0),
            head=config.get("head", "flatten"),
            head_hidden=config.get("head_hidden", 128),
        )
    raise ValueError(f"unknown model {config['model']!r}")


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Returns (loss, accuracy). Calls model.eval() itself - as in Lab 6-1.

    That matters more here than it did in Lab 6: BatchNorm, like Dropout,
    behaves differently in train and eval mode (Section 5.3).
    """
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        total_loss += criterion(out, y).item() * len(y)
        correct += (out.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def build_optimizer(name, params, lr, weight_decay):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"unknown optimizer {name!r}")


def train(config, train_ds, val_ds, test_ds=None, model=None):
    """Pass `model` to train something build_model doesn't know about -
    which is exactly how Lab 9 reuses this file for a Vision Transformer."""
    torch.manual_seed(config.get("seed", 0))
    device = get_device()

    model = (model if model is not None else build_model(config)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(config.get("optimizer", "adam"), model.parameters(),
                                config.get("lr", 1e-3), config.get("weight_decay", 0.0))

    epochs = config.get("epochs", 10)
    scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
                 if config.get("scheduler") == "cosine" else None)

    bs = config.get("batch_size", 128)
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=512)

    run_name = config["run_name"]
    writer = SummaryWriter(os.path.join("runs", run_name))
    ckpt_dir = os.path.join("checkpoints", run_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, "best.pt")

    # Log one batch of inputs so you can SEE what the model is being fed -
    # augmentation bugs are far easier to spot in a picture than in a number.
    sample, _ = next(iter(train_loader))
    writer.add_images("inputs/train", sample[:32] * 0.3081 + 0.1307, 0)

    best_val, best_epoch, bad = float("inf"), 0, 0
    history, t0 = [], time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * len(y)
            correct += (out.argmax(dim=1) == y).sum().item()
            n += len(y)
        if scheduler is not None:
            scheduler.step()

        train_loss, train_acc = run_loss / n, correct / n
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
        writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
        writer.add_scalar("generalization_gap", train_acc - val_acc, epoch)
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)
        history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                            val_loss=val_loss, val_acc=val_acc))

        if val_loss < best_val:
            best_val, best_epoch, bad = val_loss, epoch, 0
            torch.save({"model_state_dict": model.state_dict(), "config": config,
                        "epoch": epoch, "val_loss": val_loss, "val_acc": val_acc},
                       ckpt_path)
        else:
            bad += 1
            if bad >= config.get("patience", 3):
                break

    elapsed = time.perf_counter() - t0
    stopped = epoch

    # Test with the BEST weights, not whatever the last epoch happened to leave.
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    result = dict(run_name=run_name, best_epoch=best_epoch, stopped_epoch=stopped,
                  best_val_loss=best_val, val_acc=ckpt["val_acc"],
                  train_time=elapsed, n_params=count_params(model), history=history)
    if test_ds is not None:
        test_loader = DataLoader(test_ds, batch_size=512)
        result["test_loss"], result["test_acc"] = evaluate(
            model, test_loader, criterion, device)

    writer.add_hparams({k: str(v) for k, v in config.items()},
                       {"hparam/val_acc": ckpt["val_acc"],
                        "hparam/test_acc": result.get("test_acc", float("nan"))})
    writer.close()
    return result

---
## 4️⃣ Worked Example: MLP vs CNN, Same Data, Same Loop

The controlled comparison Lab 6-2 set up and could not finish.

In [ ]:
"""run_cnn_example.py - the comparison Lab 6-2 pointed at."""

import torch
from torch.utils.data import random_split

from img_dataset import MNISTImageDataset
from img_train import train

full_train = MNISTImageDataset(train=True)
test_ds = MNISTImageDataset(train=False)
val_ds, train_ds = random_split(
    full_train, [6000, len(full_train) - 6000],
    generator=torch.Generator().manual_seed(42))

base = dict(epochs=8, patience=3, batch_size=128, optimizer="adam", lr=1e-3)

runs = {
    "mlp_128":    dict(model="mlp", hidden=(128,)),
    "mlp_512":    dict(model="mlp", hidden=(512, 256)),
    "cnn_small":  dict(model="cnn", channels=(16,)),
    "cnn_32_64":  dict(model="cnn", channels=(32, 64)),
    "cnn_gap":    dict(model="cnn", channels=(32, 64), head="gap"),
}

from cnn_model import SimpleCNN, count_params

print(f"{'run':12s} {'conv':>8s} {'head':>8s} {'total':>9s} {'test_acc':>9s} {'time_s':>8s}")
for name, cfg in runs.items():
    res = train({**base, "run_name": f"ex_{name}", **cfg}, train_ds, val_ds, test_ds)
    # Split the parameter count: how much is feature extraction, how much is
    # the classifier on top? For a CNN these are wildly different numbers.
    if cfg["model"] == "cnn":
        m = SimpleCNN(channels=tuple(cfg["channels"]), head=cfg.get("head", "flatten"))
        conv = sum(p.numel() for p in m.features.parameters())
        head = sum(p.numel() for p in m.classifier.parameters())
    else:
        conv, head = 0, res["n_params"]
    print(f"{name:12s} {conv:8d} {head:8d} {res['n_params']:9d} "
          f"{res['test_acc']:9.4f} {res['train_time']:8.1f}")

### What to look for

- The CNN beats both MLPs.
- **Read the `conv` and `head` columns.** `cnn_small`'s convolution is ~160 parameters; its
  head is ~400,000. The feature extractor is nearly free; the classifier is nearly
  everything. Any "fewer parameters" claim has to say *which* parameters.
- `cnn_gap` is ~15× smaller but loses several points — averaging over the 7×7 map discards
  *where* each feature fired, and for digit shapes position matters.
- The margin on plain MNIST is **modest**. MNIST is nearly saturated. Section 5.4 is where
  the architectures actually separate.

---
## 5️⃣ Experiments

### 5.1 Kernel size, depth, head

In [ ]:
import numpy as np
from img_train import train
from cnn_model import SimpleCNN, receptive_field, count_params

sweeps = {
    "k3":        dict(model="cnn", channels=(32, 64), kernel_size=3),
    "k5":        dict(model="cnn", channels=(32, 64), kernel_size=5),
    "k7":        dict(model="cnn", channels=(32, 64), kernel_size=7),
    "c16":       dict(model="cnn", channels=(16,)),
    "c32_64":    dict(model="cnn", channels=(32, 64)),
    "c32_64_128": dict(model="cnn", channels=(32, 64, 128)),
    "gap":       dict(model="cnn", channels=(32, 64), head="gap"),
}

print(f"{'run':12s} {'params':>9s} {'RF':>4s} {'test_acc':>9s} {'time_s':>8s}")
for name, cfg in sweeps.items():
    res = train({**base, "run_name": f"s51_{name}", **cfg}, train_ds, val_ds, test_ds)
    m = SimpleCNN(channels=tuple(cfg["channels"]),
                  kernel_size=cfg.get("kernel_size", 3),
                  head=cfg.get("head", "flatten"))
    print(f"{name:12s} {res['n_params']:9d} {receptive_field(m.layer_spec):4d} "
          f"{res['test_acc']:9.4f} {res['train_time']:8.1f}")

### 5.2 Pooling vs strided convolution

Both halve the spatial size; max-pooling has no parameters, a stride-2 convolution *learns*
how to downsample.

In [ ]:
for ds in ("max", "avg", "stride"):
    accs = []
    for seed in (0, 1, 2):
        res = train({**base, "run_name": f"s52_{ds}_s{seed}", "model": "cnn",
                     "channels": (32, 64), "downsample": ds, "seed": seed},
                    train_ds, val_ds, test_ds)
        accs.append(res["test_acc"])
    print(f"downsample={ds:7s} test_acc={np.mean(accs):.4f} +/- {np.std(accs):.4f}")

### 5.3 BatchNorm — and `model.eval()` for the second time

BatchNorm normalises using the **current batch's** statistics during training and a running
average at eval time. That makes it the second layer — after Dropout in Lab 6-2 — whose
behaviour depends on `train()` vs `eval()`.

In [ ]:
for norm in ("none", "batch"):
    for lr in (1e-3, 1e-2):
        res = train({**base, "run_name": f"s53_{norm}_lr{lr}", "model": "cnn",
                     "channels": (32, 64), "norm": norm, "lr": lr},
                    train_ds, val_ds, test_ds)
        # epoch at which validation accuracy first reached 98%
        reached = next((h["epoch"] for h in res["history"] if h["val_acc"] >= 0.98), None)
        print(f"norm={norm:5s} lr={lr:<6} test_acc={res['test_acc']:.4f} "
              f"first_epoch_at_98%={reached}")

**The `eval()` bug on BatchNorm.** Write a helper that does *not* call `evaluate()`
(which calls `model.eval()` itself and would repair the bug silently — the Lab 6-2 trap).

In [ ]:
import torch
from torch.utils.data import DataLoader
from img_train import build_model


@torch.no_grad()
def acc_leaving_mode_alone(model, ds, batch_size=512):
    """Does NOT call model.eval() - that is the entire point."""
    correct = n = 0
    for x, y in DataLoader(ds, batch_size=batch_size):
        correct += (model(x).argmax(dim=1) == y).sum().item()
        n += len(y)
    return correct / n


cfg = {**base, "run_name": "s53_bn_demo", "model": "cnn",
       "channels": (32, 64), "norm": "batch"}
res = train(cfg, train_ds, val_ds, test_ds)
model = build_model(cfg)
model.load_state_dict(torch.load("checkpoints/s53_bn_demo/best.pt",
                                 map_location="cpu")["model_state_dict"])

model.train()
print("train() mode, bs=512:", [round(acc_leaving_mode_alone(model, test_ds), 4) for _ in range(3)])
model.eval()
print("eval()  mode, bs=512:", [round(acc_leaving_mode_alone(model, test_ds), 4) for _ in range(3)])

# The nastiest version: train() mode with batch size 1 normalises each sample
# by its OWN statistics, which destroys the signal entirely.
small = torch.utils.data.Subset(test_ds, range(200))
model.train()
print("train() mode, bs=1  :", round(acc_leaving_mode_alone(model, small, batch_size=1), 4))
model.eval()
print("eval()  mode, bs=1  :", round(acc_leaving_mode_alone(model, small, batch_size=1), 4))

### 5.4 Data augmentation and the shifted test set

**This is the experiment that shows what convolution is for.** MNIST digits are centred, so
a model can score well by memorising absolute positions. Build a test set where that stops
working.

In [ ]:
from img_dataset import MNISTImageDataset
from img_train import evaluate
from torch import nn

train_plain = MNISTImageDataset(train=True)
train_aug = MNISTImageDataset(train=True, shift_max=4)                    # augmentation
test_plain = MNISTImageDataset(train=False)
test_shift = MNISTImageDataset(train=False, shift_max=4, fixed_shift=True)  # fixed

g = torch.Generator().manual_seed(42)
val_p, trn_p = random_split(train_plain, [6000, len(train_plain) - 6000], generator=g)
g = torch.Generator().manual_seed(42)
val_a, trn_a = random_split(train_aug, [6000, len(train_aug) - 6000], generator=g)

criterion, device = nn.CrossEntropyLoss(), torch.device("cpu")
print(f"{'model':6s} {'trained on':11s} {'plain':>8s} {'shifted':>8s} {'drop':>8s}")
for mname, mcfg in [("mlp", dict(model="mlp", hidden=(512, 256))),
                    ("cnn", dict(model="cnn", channels=(32, 64)))]:
    for tname, (trn, val) in [("plain", (trn_p, val_p)), ("augmented", (trn_a, val_a))]:
        cfg = {**base, "run_name": f"s54_{mname}_{tname}", **mcfg}
        train(cfg, trn, val, test_plain)
        m = build_model(cfg)
        m.load_state_dict(torch.load(f"checkpoints/s54_{mname}_{tname}/best.pt",
                                     map_location="cpu")["model_state_dict"])
        a = evaluate(m, DataLoader(test_plain, batch_size=512), criterion, device)[1]
        b = evaluate(m, DataLoader(test_shift, batch_size=512), criterion, device)[1]
        print(f"{mname:6s} {tname:11s} {a:8.4f} {b:8.4f} {a - b:+8.4f}")

**Task.** Which cell drops the most? Which model degrades most gracefully *without*
augmentation? Convolution is translation **equivariant** — so why does an un-augmented CNN
still lose accuracy here? (Think about what the `flatten` head does to a shifted feature
map, then try `head="gap"` to test your explanation.)

Also open the **IMAGES** tab in TensorBoard and look at `inputs/train` for the augmented
run — do the shifted digits look right, or is there a visible border artifact?

### 5.5 What the filters learn

In [ ]:
import matplotlib.pyplot as plt
import torch

from cnn_model import SimpleCNN

ckpt = torch.load("checkpoints/ex_cnn_32_64/best.pt", map_location="cpu")
model = SimpleCNN(channels=(32, 64))
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# --- first-layer kernels: 32 filters of 1x3x3 ---
w = model.features[0].weight.detach()            # (32, 1, 3, 3)
fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for ax, kernel in zip(axes.ravel(), w):
    ax.imshow(kernel[0], cmap="gray")
    ax.axis("off")
fig.suptitle("Layer-1 kernels")
plt.tight_layout()
plt.show()

In [ ]:
# Feature maps after each block, for one test digit.
x, y = test_ds[0]
maps = model.feature_maps(x[None]) if hasattr(model, "feature_maps") else []

for depth, fm in enumerate(maps):
    n_show = min(8, fm.shape[1])
    fig, axes = plt.subplots(1, n_show + 1, figsize=(2 * (n_show + 1), 2.2))
    axes[0].imshow(x[0], cmap="gray")
    axes[0].set_title("input")
    axes[0].axis("off")
    for ch, ax in enumerate(axes[1:]):
        ax.imshow(fm[0, ch], cmap="viridis")
        ax.axis("off")
    fig.suptitle(f"after block {depth + 1}   shape={tuple(fm.shape[1:])}")
    plt.tight_layout()
    plt.show()

### 5.6 Overfitting on a small subset

As in Lab 6-2, full MNIST barely overfits. Subsample to make the effect visible, then ask
which regulariser actually helps: dropout, weight decay, or more data via augmentation.

In [ ]:
small_train = MNISTImageDataset(train=True, limit=1000)
small_aug = MNISTImageDataset(train=True, limit=1000, shift_max=4)

trials = {
    "plain":      (small_train, dict()),
    "dropout0.25": (small_train, dict(dropout=0.25)),
    "dropout0.5": (small_train, dict(dropout=0.5)),
    "wd1e-2":     (small_train, dict(weight_decay=1e-2)),
    "augmented":  (small_aug, dict()),
}

print(f"{'setting':12s} {'train_acc':>10s} {'test_acc':>9s} {'gap':>7s}")
for name, (ds, extra) in trials.items():
    res = train({**base, "run_name": f"s56_{name}", "model": "cnn",
                 "channels": (32, 64), **extra}, ds, val_ds, test_ds)
    last = res["history"][-1]
    print(f"{name:12s} {last['train_acc']:10.4f} {res['test_acc']:9.4f} "
          f"{last['train_acc'] - res['test_acc']:+7.4f}")

---
## 6️⃣ Inference

> ⚠️ **MNIST is white-on-black**, centred, ~20 px tall. A digit you draw or photograph is
> usually black-on-white and will be confidently misclassified. Invert it
> (`gray = 255 - gray`) and centre it. This preprocessing mismatch — not the model — is the
> usual reason "it works on the test set but not on my own digit".

In [ ]:
# Make a sample digit so this cell runs top-to-bottom. For the exercise,
# replace my_digit.png with one you drew or photographed yourself.
import os

import cv2

if not os.path.exists("my_digit.png"):
    from img_dataset import load_mnist_arrays
    imgs, tgts = load_mnist_arrays(train=False)
    cv2.imwrite("my_digit.png", imgs[0])
    print("wrote a sample my_digit.png, true label =", int(tgts[0]))


"""predict_digit.py - classify one image with a saved checkpoint."""

import cv2
import numpy as np
import torch

from cnn_model import SimpleCNN
from img_dataset import MNIST_MEAN, MNIST_STD

CHECKPOINT_PATH = "checkpoints/ex_cnn_32_64/best.pt"   # hardcoded - change
IMAGE_PATH = "my_digit.png"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
config = ckpt["config"]
print(f"Loaded epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.4f})")

model = SimpleCNN(
    channels=tuple(config.get("channels", (32, 64))),
    kernel_size=config.get("kernel_size", 3),
    norm=config.get("norm", "none"),
    downsample=config.get("downsample", "max"),
    head=config.get("head", "flatten"),
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()                 # disables dropout AND switches BatchNorm to running stats

gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
if gray is None:
    raise FileNotFoundError(IMAGE_PATH)
gray = cv2.resize(gray, (28, 28), interpolation=cv2.INTER_AREA)

# Preprocess EXACTLY as training did: scale to [0,1], then normalize.
x = torch.tensor(gray, dtype=torch.float32)[None, None] / 255.0
x = ((x - MNIST_MEAN) / MNIST_STD).to(device)

with torch.no_grad():
    probs = torch.softmax(model(x), dim=1)[0]   # softmax HERE, not in forward()

pred = int(probs.argmax())
print(f"Predicted: {pred}  (confidence {probs[pred]:.3f})")
for score, idx in zip(*[t.tolist() for t in torch.topk(probs, 3)]):
    print(f"  {idx}: {score:.3f}")

---
## 🧪 Student Exercise (Evaluative)

Full description in [`labsheet8.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet8.md).

**Part 1 — architecture grid:** `channels` × `downsample` × `norm` = 12 configs × 3 seeds,
with Lab 6's MLPs retrained here as controls.

**Part 2 — robustness:** the 4×2 table above, extended with a **rotated** (±15°) test
column, plus a parameter-matched MLP/CNN pair.

**Part 3 — data efficiency:** both models at `limit ∈ {500, 2000, 10000, 60000}`, test
accuracy vs training-set size on a log x-axis.

**Key analysis question:** *where does the CNN's advantage actually come from?* Design a
comparison separating **parameter sharing** from **parameter count** — the `conv`/`head`
split in Section 4 is where that question starts.

In [ ]:
# Starter for Part 3 - data efficiency.
import numpy as np
import matplotlib.pyplot as plt

sizes = [500, 2000, 10000, 54000]
curves = {"mlp": [], "cnn": []}

for n in sizes:
    sub = MNISTImageDataset(train=True, limit=n)
    for name, cfg in [("mlp", dict(model="mlp", hidden=(512, 256))),
                      ("cnn", dict(model="cnn", channels=(32, 64)))]:
        res = train({**base, "run_name": f"eff_{name}_{n}", **cfg}, sub, val_ds, test_ds)
        curves[name].append(res["test_acc"])
        print(f"n={n:6d} {name}: {res['test_acc']:.4f}")

for name, ys in curves.items():
    plt.plot(sizes, ys, marker="o", label=name)
plt.xscale("log")
plt.xlabel("training images")
plt.ylabel("test accuracy")
plt.title("Data efficiency: does the spatial prior matter most when data is scarce?")
plt.legend()
plt.grid(alpha=0.3)
plt.show()